# ML Optimization for GPU Selection

This notebook uses predictive model outputs as objective surfaces for constrained GPU selection across the observed Vulkan profiles.

In [1]:
import pandas as pd
from IPython.display import display

from src.ml_model_utils import (
    reduce_feature_set,
    regression_target,
    run_sklearn_regression_models,
    run_tensorflow_regressor,
    run_torch_regressor,
)
from src.vulkan_math_data import feature_matrix, load_ml_dataset

2026-06-02 17:23:59.713247: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-02 17:24:00.133443: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-02 17:24:00.312489: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-02 17:24:00.313904: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-02 17:24:00.590060: I tensorflow/core/platform/cpu_feature_gua

In [2]:
ml_frame, diagnostics = load_ml_dataset()
X, _ = feature_matrix(ml_frame)
X = reduce_feature_set(X, max_features=16)
y, target_name = regression_target(ml_frame)

sklearn_runs = run_sklearn_regression_models(X, y)
torch_run = run_torch_regressor(X, y, epochs=50)
tensorflow_run = run_tensorflow_regressor(X, y, epochs=50)
baseline_run = sklearn_runs[0]

In [3]:
candidate_table = ml_frame[
    [
        "report_id",
        "normalized_gpu_name",
        "agg__descriptor_feature_count",
        "agg__raytracing_feature_count",
        "agg__extension_count",
    ]
].copy()
candidate_table["score__sklearn"] = baseline_run.predictions
candidate_table["score__pytorch"] = torch_run.predictions
candidate_table["score__tensorflow"] = tensorflow_run.predictions
display(candidate_table)

,report_id,normalized_gpu_name,agg__descriptor_feature_count,agg__raytracing_feature_count,agg__extension_count,score__sklearn,score__pytorch,score__tensorflow
0,49077,rtx 4070 ti,41,39,0,424.143939,27.297777,35.315392
1,49118,ryzen 9 7900x 12 core processor,41,36,0,369.356061,23.608957,54.629101


In [4]:
descriptor_threshold = candidate_table["agg__descriptor_feature_count"].median()
ray_threshold = candidate_table["agg__raytracing_feature_count"].median()
extension_threshold = candidate_table["agg__extension_count"].median()

feasible = candidate_table[
    (candidate_table["agg__descriptor_feature_count"] >= descriptor_threshold)
    & (candidate_table["agg__raytracing_feature_count"] >= ray_threshold)
    & (candidate_table["agg__extension_count"] >= extension_threshold)
].copy()

print("Descriptor threshold:", descriptor_threshold)
print("Ray threshold:", ray_threshold)
print("Extension threshold:", extension_threshold)
display(feasible.sort_values("score__sklearn", ascending=False))

Descriptor threshold: 41.0
Ray threshold: 37.5
Extension threshold: 0.0


,report_id,normalized_gpu_name,agg__descriptor_feature_count,agg__raytracing_feature_count,agg__extension_count,score__sklearn,score__pytorch,score__tensorflow
0,49077,rtx 4070 ti,41,39,0,424.143939,27.297777,35.315392


In [5]:
if not feasible.empty:
    best_by_framework = {
        "scikit-learn": feasible.sort_values("score__sklearn", ascending=False).iloc[0][
            "report_id"
        ],
        "PyTorch": feasible.sort_values("score__pytorch", ascending=False).iloc[0][
            "report_id"
        ],
        "TensorFlow": feasible.sort_values("score__tensorflow", ascending=False).iloc[
            0
        ]["report_id"],
    }
    display(
        pd.DataFrame(
            best_by_framework.items(), columns=["framework", "selected_report_id"]
        )
    )
else:
    print("No feasible candidate satisfied the current constraints.")

,framework,selected_report_id
0,scikit-learn,49077
1,PyTorch,49077
2,TensorFlow,49077
